# Bayes' theorem

Bayes' theorem is the rule for changing your mind by the right amount when evidence arrives. Start with a belief, see data, end with an updated belief. The theorem says exactly how much to update, no more and no less. It is also the basis of the naive Bayes classifier built later in this notebook, and the same machinery explains regularisation in notebook 03.

Order: intuition, maths, idea, from scratch, library, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

## 1. Intuition

A test for a rare disease is 90% accurate. You test positive. How worried should you be?

Most people say 90%, and most are badly wrong. The honest answer depends on how rare the disease is, and it is usually far lower than the test's accuracy suggests. The healthy majority is so large that even a small false-positive rate produces a flood of false alarms that swamps the few true positives. The prior, how common the disease is before any test, matters just as much as the evidence from the test. Ignoring the prior is the most common probability mistake, and the numbers below make it concrete.

## 2. Maths

$$P(H\mid D)=\frac{P(D\mid H)\,P(H)}{P(D)},\qquad P(D)=\sum_h P(D\mid h)P(h)$$

- $P(H)$ is the prior: what you believe before seeing the data.
- $P(D\mid H)$ is the likelihood: how well the hypothesis explains the data you saw.
- $P(H\mid D)$ is the posterior: what you believe after folding the data in.
- $P(D)$ is the evidence, a normaliser that makes the posteriors over all hypotheses sum to 1.

Strip away the normaliser and the rule is one line to remember: posterior is proportional to likelihood times prior. Updating a belief is multiplying by how well the data fits.

## 3. The base-rate example

Prevalence 1%. Sensitivity (positive given sick) 90%. False positive rate (positive given healthy) 9%.

In [ ]:
prior = 0.01
sens = 0.90
fpr = 0.09

p_pos = sens * prior + fpr * (1 - prior)
posterior = sens * prior / p_pos
print("P(positive)         :", p_pos)
print("P(sick | positive)  :", posterior)

# Check by simulating 1,000,000 people
n = 1_000_000
sick = rng.random(n) < prior
positive = rng.random(n) < np.where(sick, sens, fpr)
print("simulated           :", sick[positive].mean())
print("sick among positives:", sick[positive].sum(), "of", positive.sum())

Out of about 98,000 positives, only 9,000 are actually sick. A positive result does move the needle, lifting the chance from the 1% base rate to about 9%, a ninefold jump. But it lands nowhere near the 90% the test's accuracy makes people expect, because the 99% of healthy people contribute far more false positives than the 1% sick contribute true ones.

Evidence compounds, though. A second independent positive test uses the first posterior as its new prior, so two positives together are much stronger than one.

In [ ]:
def update(prior: float, sens: float, fpr: float) -> float:
    return sens * prior / (sens * prior + fpr * (1 - prior))


p = prior
for i in range(1, 4):
    p = update(p, sens, fpr)
    print(f"after {i} positive test(s): {p:.3f}")

## 4. Beta-Binomial: updating a belief about a coin

Now a continuous unknown. A coin has some bias $\theta$ between 0 and 1 that you cannot see. Put a prior on it, $\theta\sim\mathrm{Beta}(a,b)$, a tunable curve over the interval. Observe $k$ heads in $n$ flips. The likelihood is $\theta^k(1-\theta)^{n-k}$ and the prior is proportional to $\theta^{a-1}(1-\theta)^{b-1}$. Multiply them, as Bayes says, and the $\theta$ powers add:

$$P(\theta\mid k,n)\propto\theta^{a+k-1}(1-\theta)^{b+n-k-1}=\mathrm{Beta}(a+k,\,b+n-k)$$

The posterior is a Beta again, the same family as the prior. A prior with this property is a conjugate prior, and it makes updating trivial: just add the counts. Read $a$ and $b$ as pseudo-counts, heads and tails you pretend to have seen before any real flip, which encode how strong your belief is. The posterior mean is $(a+k)/(a+b+n)$.

In [ ]:
def beta_update(a: float, b: float, k: int, n: int) -> tuple[float, float]:
    return a + k, b + n - k


a, b = 2, 2  # weak prior centred on 0.5
k, n = 7, 10  # 7 heads in 10 flips
a_post, b_post = beta_update(a, b, k, n)
print("posterior: Beta", (a_post, b_post), " mean:", a_post / (a_post + b_post))

# Check against a brute-force grid: prior * likelihood, normalised
theta = np.linspace(0.001, 0.999, 999)
unnorm = stats.beta.pdf(theta, a, b) * theta**k * (1 - theta) ** (n - k)
grid_post = unnorm / (unnorm.sum() * (theta[1] - theta[0]))
print("max gap to grid:", np.abs(grid_post - stats.beta.pdf(theta, a_post, b_post)).max())

plt.figure(figsize=(6, 3))
plt.plot(theta, stats.beta.pdf(theta, a, b), label="prior Beta(2, 2)")
plt.plot(theta, stats.beta.pdf(theta, a_post, b_post), label="posterior Beta(9, 5)")
plt.axvline(k / n, color="grey", ls=":", label="MLE 0.7")
plt.legend()
plt.show()

## 5. More data overwhelms the prior

Two people start with different priors about a coin whose true bias is 0.7: one sceptic who is sure it is fair, Beta(50, 50), and one with no opinion, Beta(1, 1). Feed them the same flips.

In [ ]:
true_theta = 0.7
flips = rng.random(1000) < true_theta
priors = {"sceptic Beta(50,50)": (50, 50), "flat Beta(1,1)": (1, 1)}

print(f"{'n':>5} " + " ".join(f"{name:>22}" for name in priors))
for n in (0, 10, 50, 200, 1000):
    k = int(flips[:n].sum())
    means = []
    for a0, b0 in priors.values():
        a1, b1 = beta_update(a0, b0, k, n)
        means.append(a1 / (a1 + b1))
    print(f"{n:>5} " + " ".join(f"{m:>22.3f}" for m in means))

The sceptic starts dead certain at 0.5 and still reaches 0.72 after 1000 flips, within 0.03 of the open-minded flat prior at 0.74 (the true bias is 0.7, and the gap is just noise in this run). The sceptic's Beta(50, 50) prior is worth 100 pseudo-flips, so it only fades once the real count climbs well past that. At 10 real flips it still rules: 0.52 for the sceptic against 0.67 for the flat prior. Enough data overwhelms any prior, but a stubborn prior needs a lot of it.

## 6. Naive Bayes classifier

Bayes' theorem becomes a classifier when the hypothesis is a class label. Pick the class $c$ with the highest posterior given the features $x_1,\dots,x_d$:

$$P(c\mid x)\propto P(c)\prod_{j=1}^d P(x_j\mid c)$$

"Naive" names the shortcut: the features are treated as independent given the class. That assumption is usually false, yet it turns one impossibly hard joint density into $d$ easy one-dimensional ones, and the classifier often works well anyway. For continuous features, model each one with a Gaussian per class, its mean and variance estimated from that class's rows. Work in log space so the product of many small probabilities does not underflow:

$$\log P(c\mid x)=\log P(c)+\sum_j\left[-\tfrac12\log(2\pi\sigma_{cj}^2)-\frac{(x_j-\mu_{cj})^2}{2\sigma_{cj}^2}\right]+\text{const}$$

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


class GaussianNaiveBayes:
    def __init__(self, var_smoothing: float = 1e-9) -> None:
        self.var_smoothing = var_smoothing

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GaussianNaiveBayes":
        self.classes_ = np.unique(y)
        self.log_prior_ = np.log(np.array([(y == c).mean() for c in self.classes_]))
        self.mean_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        self.var_ = np.array([X[y == c].var(axis=0) for c in self.classes_])
        # Same small floor sklearn uses, so a zero-variance feature does not divide by zero
        self.var_ += self.var_smoothing * X.var(axis=0).max()
        return self

    def log_joint(self, X: np.ndarray) -> np.ndarray:
        out = []
        for i in range(len(self.classes_)):
            ll = -0.5 * np.log(2 * np.pi * self.var_[i]) - (X - self.mean_[i]) ** 2 / (
                2 * self.var_[i]
            )
            out.append(self.log_prior_[i] + ll.sum(axis=1))
        return np.array(out).T

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        lj = self.log_joint(X)
        lj -= lj.max(axis=1, keepdims=True)  # stabilise before exponentiating
        p = np.exp(lj)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.classes_[self.log_joint(X).argmax(axis=1)]


mine = GaussianNaiveBayes().fit(X_train, y_train)
print("accuracy:", (mine.predict(X_test) == y_test).mean())

## 7. Library: sklearn GaussianNB

The two should agree on predictions and on probabilities.

In [ ]:
sk = GaussianNB().fit(X_train, y_train)
print("accuracy:", sk.score(X_test, y_test))
print("same predictions:", (sk.predict(X_test) == mine.predict(X_test)).all())
print("max probability gap:", np.abs(sk.predict_proba(X_test) - mine.predict_proba(X_test)).max())

## 8. What breaks

In [ ]:
# 1. Zero counts. Word counts for a spam class: the word "prize" never appeared in ham.
ham_count, ham_total, vocab = 0, 1000, 5000
print("P(prize | ham), raw      :", ham_count / ham_total)  # one zero wipes out the whole product
print("P(prize | ham), Laplace  :", (ham_count + 1) / (ham_total + vocab))

# 2. Correlated features are counted twice. Duplicate one iris column and watch the
# probabilities get more extreme even though no new information was added.
X_dup = np.hstack([X_train, X_train[:, [2]]])
dup = GaussianNaiveBayes().fit(X_dup, y_train)
row = int(mine.predict_proba(X_test).max(axis=1).argmin())  # the least certain test flower
print("single copy:", mine.predict_proba(X_test[[row]]))
print("duplicated :", dup.predict_proba(np.hstack([X_test, X_test[:, [2]]])[[row]]))

A zero count turns one strong piece of evidence into an absolute veto: a single $P(x_j\mid c)=0$ zeroes the whole product no matter what the other features say. The fix is to add one pseudo-count to every cell, called Laplace smoothing, which is the Beta(1, 1) prior from earlier under a different name. The repeated-feature failure is quieter. Counting correlated features as independent double-counts their evidence and pushes the probabilities towards 0 and 1. The predicted class often survives because the ranking is unchanged, but the probabilities can no longer be trusted as calibrated.

Next: where loss functions come from, and why a prior is the same thing as regularisation.